# Model Embeddings 

In [2]:
import pandas as pd 
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
df = pd.read_csv("D:\Machine Learning\Datasets\\titles.csv")
df.head()
# df.columns

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\dbda.STUDENTSDC\AppData\Local\Temp\ipykernel_8756\1080905782.py:1: SyntaxWarning: invalid escape sequence '\M'
  df = pd.read_csv("D:\Machine Learning\Datasets\\titles.csv")


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [4]:
df_indian_movies = df[(df['production_countries'].str.contains('IN')) 
& (df['type'] == 'MOVIE') & (df['release_year'] >= 1980)]
df_indian_movies.head()
# df_indian_movies.shape

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5


In [5]:
df_indian_movies.loc[:,'info'] = "Title: " 
+ df_indian_movies['title'].copy()+ 
", Description: "+ df_indian_movies['description'].copy()
df_indian_movies

SyntaxError: invalid syntax (2476488912.py, line 2)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)
# [4, 4]

d:\Anaconda\envs\hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Anaconda\envs\hf\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dbda.STUDENTSDC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer

torch.Size([4, 4])


In [ ]:
from tqdm import tqdm
all_desc = list(df_indian_movies['info'])
out_embs = []
for i in tqdm(all_desc):
    embd = model.encode(i)
    out_embs.append(embd)
out_embs = np.array(out_embs)
out_embs.shape

100%|██████████| 570/570 [00:07<00:00, 75.78it/s]


(570, 384)

In [ ]:
def search_similar(query):
    qy_emb = model.encode([query])
    sims = []
    for e in tqdm(out_embs):
        sims.append(model.similarity(qy_emb, e).numpy()[0])
    df_ind = df_indian_movies[['id', 'title', 'description']].copy()
    df_ind.loc[:,'sims'] = np.array(sims)
    return df_ind.sort_values('sims', ascending=False)[:5]

In [ ]:
search_similar("three college friends do a lot of fun")

  0%|          | 0/570 [00:00<?, ?it/s]

100%|██████████| 570/570 [00:00<00:00, 7884.39it/s]


,id,title,description,sims
434,tm61056,Dil Chahta Hai,Three inseparable childhood friends are just o...,0.455433
721,tm182125,Zindagi Na Milegi Dobara,Three friends who were inseparable in childhoo...,0.401727
5154,tm1213009,Dear Friend,Four close friends who live together in Bangal...,0.376396
316,tm142564,3 Idiots,Rascal. Joker. Dreamer. Genius... You've never...,0.374349
972,tm185797,Fukrey,The paths of four dream-chasing college friend...,0.373624


In [6]:
cred = pd.read_csv("D:\Machine Learning\Datasets\credits.csv")
cred.head()

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\dbda.STUDENTSDC\AppData\Local\Temp\ipykernel_8756\1656382350.py:1: SyntaxWarning: invalid escape sequence '\M'
  cred = pd.read_csv("D:\Machine Learning\Datasets\credits.csv")


,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR


In [7]:
cred['role'].unique()

array(['ACTOR', 'DIRECTOR'], dtype=object)

In [ ]:
df_indian_movies = df[(df['production_countries'].str.contains('IN')) 
& (df['type'] == 'MOVIE') & (df['release_year'] >= 1980)]
df_indian_movies.head()
# df_indian_movies.shape